# Model 02: structured populations and migration

This notebook divides the population into demes and lets genealogical ancestry move between them through parental-source mixing. It still tracks **genealogical ancestry only**, not DNA.

A hard barrier can disconnect the chain. This lets us test when the complete-mixing intuition from Model 01 fails.

In [ ]:
import os, sys, subprocess
from pathlib import Path

try:
    import google.colab  # type: ignore
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

REPO_DIR = None
if IN_COLAB:
    REPO_DIR = Path('/content/Evolution-Creation')
    if not (REPO_DIR / '.git').exists():
        subprocess.run(['git','clone','-q','https://github.com/vafaei-ar/Evolution-Creation.git',str(REPO_DIR)],check=True)
    else:
        subprocess.run(['git','-C',str(REPO_DIR),'fetch','-q','origin','main'],check=True)
        subprocess.run(['git','-C',str(REPO_DIR),'checkout','-q','main'],check=True)
        subprocess.run(['git','-C',str(REPO_DIR),'reset','--hard','origin/main'],check=True)
    subprocess.run([sys.executable,'-m','pip','install','-q','-e',f'{REPO_DIR}[dev]'],check=True)
else:
    for candidate in (Path.cwd(), Path.cwd().parent):
        if (candidate / 'src' / 'evolution_creation').exists():
            REPO_DIR = candidate
            break

if REPO_DIR is not None:
    src_path = str(REPO_DIR / 'src')
    if src_path not in sys.path:
        sys.path.insert(0, src_path)

print('Environment ready:', REPO_DIR if REPO_DIR is not None else 'using installed Python environment')


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import ipywidgets as widgets
from IPython.display import display
from evolution_creation.structured import (
    add_linear_barrier,
    make_linear_migration_matrix,
    simulate_structured_replicates,
)


## Controls

`migration rate` is the total probability that a parent comes from an adjacent deme. `barrier after` uses 1-based labels for the interface shown to the user. Set it to `None` for a connected chain.

In [ ]:
n_demes = widgets.IntSlider(value=5, min=2, max=8, step=1, description='Demes')
deme_size = widgets.IntSlider(value=300, min=50, max=1000, step=50, description='Size/deme')
migration_rate = widgets.FloatSlider(value=0.02, min=0.0, max=0.20, step=0.001, readout_format='.3f', description='Migration')
generations = widgets.IntSlider(value=80, min=5, max=200, step=5, description='Generations')
founder_count = widgets.IntSlider(value=10, min=1, max=50, step=1, description='Founders')
replicates = widgets.IntSlider(value=100, min=10, max=300, step=10, description='Replicates')
barrier_after = widgets.Dropdown(options=[('None', None)] + [(f'after deme {i}', i-1) for i in range(1, 8)], value=None, description='Barrier')
seed = widgets.IntText(value=20260920, description='Seed')
display(n_demes, deme_size, migration_rate, generations, founder_count, replicates, barrier_after, seed)

In [ ]:
def run_model(_=None):
    matrix = make_linear_migration_matrix(n_demes.value, migration_rate.value)
    if barrier_after.value is not None and barrier_after.value < n_demes.value - 1:
        matrix = add_linear_barrier(matrix, barrier_after.value)

    curves = simulate_structured_replicates(
        deme_sizes=[deme_size.value] * n_demes.value,
        generations=generations.value,
        migration_matrix=matrix,
        founder_deme=0,
        founder_count=min(founder_count.value, deme_size.value),
        replicates=replicates.value,
        seed=seed.value,
    )

    median = np.median(curves, axis=0)
    x = np.arange(generations.value + 1)
    fig, ax = plt.subplots(figsize=(9, 5))
    for d in range(n_demes.value):
        ax.plot(x, median[:, d], label=f'Deme {d+1}')
    ax.set(xlabel='Generation', ylabel='Median fraction with founder ancestry', ylim=(0, 1.02))
    ax.legend(ncol=2)
    plt.show()

    final = curves[:, -1, :]
    survived = np.any(final > 0, axis=1).mean()
    all_reached = np.all(final > 0, axis=1).mean()
    globally_fixed = np.all(final == 1, axis=1).mean()
    print(f'Founder lineage survives somewhere: {survived:.1%}')
    print(f'Every deme reached: {all_reached:.1%}')
    print(f'Global genealogical fixation: {globally_fixed:.1%}')

    fig, ax = plt.subplots(figsize=(6, 5))
    image = ax.imshow(matrix, vmin=0, vmax=1)
    ax.set(xlabel='Source deme', ylabel='Destination deme', title='Parental-source mixing matrix')
    fig.colorbar(image, ax=ax, label='Probability')
    plt.show()

run_button = widgets.Button(description='Run simulation', button_style='primary')
run_button.on_click(run_model)
display(run_button)
run_model()

## Interpretation

A fast spread in this model does not establish that historical human populations had these migration rates. A permanent hard barrier is qualitatively different from a low migration rate: with a true zero-probability connection, waiting longer cannot move ancestry across the barrier.